In [1]:
import matplotlib.pyplot as plt
%matplotlib widget
from ipywidgets import *
import numpy as np
import pandas as pd
import sys
import taurex.log
taurex.log.disableLogging()
from taurex.cache import OpacityCache,CIACache
from taurex.temperature import Guillot2010
from taurex.planet import Planet
from taurex.stellar import BlackbodyStar, PhoenixStar
from taurex.chemistry import TaurexChemistry, ConstantGas
from taurex.model import TransmissionModel
from taurex.contributions import AbsorptionContribution, CIAContribution, RayleighContribution
from taurex.binning import FluxBinner,SimpleBinner
from taurex.temperature import Isothermal
from taurex.data.spectrum.observed import ObservedSpectrum
from taurex.optimizer.nestle import NestleOptimizer

Numba not installed, using numpy instead


In [2]:
#Importing the opacity files (cross-sections and CIA)
OpacityCache().clear_cache()
OpacityCache().set_opacity_path("/ca25/ext_volume/run_daneel/Assignment3_Atmospheres/xsecs")
CIACache().set_cia_path("/ca25/ext_volume/run_daneel/Assignment3_Atmospheres/cia/hitran")

h2o_xsec = OpacityCache()['H2O']
ch4_xsec = OpacityCache()['CH4']
co2_xsec = OpacityCache()['CO2']
co_xsec = OpacityCache()['CO']

In [ ]:
# Set up the parameters and chemical abundances for the atmospheric model
planet = Planet(planet_radius=0.211,planet_mass=0.028)
star = BlackbodyStar(temperature=3457.0,radius=0.41)
chemistry = TaurexChemistry(fill_gases=['H2','He'], ratio=0.172)
chemistry.addGas(ConstantGas('H2O', mix_ratio=0.00061))
chemistry.addGas(ConstantGas('CH4', mix_ratio=0.00912))
chemistry.addGas(ConstantGas('CO2', mix_ratio=0.01778))
chemistry.addGas(ConstantGas('CO', mix_ratio=0.001))

In [ ]:
#Build an isothermal transmission model and add the contributions from physical processes
isothermal = Isothermal(T=284.0)

tm = TransmissionModel(planet=planet,
                       temperature_profile=isothermal,
                       chemistry=chemistry,
                       star=star,
                       atm_min_pressure=1e-0,
                       atm_max_pressure=1e6,
                       nlayers=30)
tm.add_contribution(AbsorptionContribution())
tm.add_contribution(CIAContribution(cia_pairs=['H2-H2','H2-He']))
tm.add_contribution(RayleighContribution())
tm.build()

#Run the model to get the transmission spectrum
res = tm.model()
res

In [ ]:
#check the keys of the fitting paramenters
list(tm.fittingParameters.keys())

In [ ]:
#Load the observed spectrum and bin it to the model resolution
obs = ObservedSpectrum('/ca25/ext_volume/run_daneel/Assignment3_Atmospheres/K2-18b_assignment3_taskA_spectrum.dat')
#Make a logarithmic grid or a linear in the wavelength
wngrid = np.sort(10000/np.linspace(0.3,13,1000))
bn = SimpleBinner(wngrid=wngrid)

bin_obs= bn.bindown(obs.wavenumberGrid, obs.spectrum)
errorbars = bn.bindown(obs.wavenumberGrid, obs.errorBar)


plt.figure()
plt.xscale('log')
plt.scatter(1e4/wngrid, bin_obs[1], label='Obs', c='b', s=1)
plt.errorbar(1e4/wngrid, bin_obs[1], yerr=errorbars[1], label='Obs_errorbars', c='k', markersize=2, alpha=0.25, elinewidth=0.25)
plt.legend()
plt.show()

In [ ]:
#Binning the native model spectrum to the observed spectrum's wavelength grid
obin = obs.create_binner()
plt.figure()
plt.scatter(obs.wavelengthGrid, obs.spectrum, label='Obs', c='b', s=1)
plt.errorbar(obs.wavelengthGrid,obs.spectrum,obs.errorBar,label='Obs_errorbars',c='k',markersize=2,alpha=0.25,elinewidth=0.25)
plt.plot(obs.wavelengthGrid,obin.bin_model(tm.model(obs.wavenumberGrid))[1],label='TM', c='r')
plt.legend()
plt.show()

In [ ]:
#Setting up the nestle optimizer for the retrieval
opt = NestleOptimizer(num_live_points=50)

#Setting up the model and observed spectrum for the optimizer
opt.set_model(tm)
opt.set_observed(obs)

#Set up which parameters to fit and their boundaries
opt.enable_fit('planet_radius')
opt.enable_fit('T')
opt.enable_fit('H2O')
opt.enable_fit('CH4')
opt.enable_fit('CO2')
opt.enable_fit('CO')
opt.set_boundary('T',[200,400])
opt.set_boundary('planet_radius',[0.1,0.3])
opt.set_boundary('H2O',[1e-8,1e-2])
opt.set_boundary('CH4',[1e-8,1e-2])
opt.set_boundary('CO2',[1e-8,1e-2])
opt.set_boundary('CO',[1e-8,1e-2])

In [ ]:
#Fit the model to the observed spectrum (quello che devi runnare Nicholas)
solution = opt.fit()
taurex.log.disableLogging()

In [ ]:
#Plot the fitted spectrum against the observed one
for solution,optimized_map,optimized_value,values in opt.get_solution():
    opt.update_model(optimized_map)
    png_plot = plt.figure()
    plt.scatter(obs.wavelengthGrid, obs.spectrum, label='Obs', c='b', s=1)
    plt.errorbar(obs.wavelengthGrid,obs.spectrum,obs.errorBar,label='Obs_errorbars',c='k',markersize=2,alpha=0.25,elinewidth=0.25)
    plt.plot(obs.wavelengthGrid,obin.bin_model(tm.model(obs.wavenumberGrid))[1],label='TM', c='r')
    plt.legend()
    plt.show()

In [ ]:
#Saving the plot of the fitted transmission spectrum
png_plot.savefig('assignment3_taskD.png')